In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.regime_switching.markov_regression import MarkovRegression

# 1. Loading and manipulating Brazilian GDP data
df_br_gdp_raw = pd.read_excel('brazil_gdp-data.xlsx')
print(df_br_gdp_raw.T.head(10)) # Finding which columns will be used
df_br_gdp = df_br_gdp_raw.T.iloc[:, [2, 4]] # Taking the right columns
df_br_gdp.columns = ['Period', 'Index']
df_br_gdp['Index'] = pd.to_numeric(df_br_gdp['Index'], errors = 'coerce')

def convert_quartely_ibge(text):
    try:
        parts = str(text).split()
        quarter = int(parts[0][0])
        year = parts[-1]
        month = (quarter * 3) - 2
        return pd.to_datetime(f"{year}-{month:02d}-01")
    except: 
        return pd.NaT

if 'Period' in df_br_gdp.columns:
    df_br_gdp['Quarter_Date'] = df_br_gdp['Period'].apply(convert_quartely_ibge)
    df_br_gdp = df_br_gdp.dropna(subset=['Quarter_Date']).set_index('Quarter_Date')

print(df_br_gdp.tail())

# 2. Converting into Log-Diff and procedure of not detecting outliers
# The reason of not detecting outlier is because the pandemic period, which the volatility is extremely high
df_br_gdp['y'] = np.log(df_br_gdp['Index']).diff()*100
df_br_gdp = df_br_gdp.replace([np.inf, -np.inf], np.nan).dropna(subset = ['y'])
std_dev = df_br_gdp['y'].std()
df_br_gdp['y'] = df_br_gdp['y'].clip(lower = -3*std_dev, upper = 3*std_dev)

# 3. Identifying 3 Regimes by Chain Markov Methodology
np.random.seed(42)
chain_markov_br_gdp = MarkovRegression(df_br_gdp['y'], k_regimes = 3, trend = 'c', switching_variance = False)
chain_markov_br_gdp.initialization = 'approximate-diffuse'
chain_markov_br_gdp_result = chain_markov_br_gdp.fit(em_iter = 50, method = 'lbfgs', search_reps = 100)
print(chain_markov_br_gdp_result.summary())

# 4. Sorting the means
# 1. Identifying Regimes by Growth Mean (const)
params_const = chain_markov_br_gdp_result.params.filter(like='const')
sorted_indices = params_const.sort_values().index

r_rec = int(sorted_indices[0].split('[')[1][0])
r_sta = int(sorted_indices[1].split('[')[1][0])
r_exp = int(sorted_indices[2].split('[')[1][0])

# 2. Calculating and Printing Expected Durations
Duration = chain_markov_br_gdp_result.expected_durations
Regime_Name_Map = {r_rec: "Recession", r_sta: "Stabilization/Stagnation", r_exp: "Expansion"}

print("\nEXPECTED DURATION OF REGIMES (IN QUARTERS):")
for i in range(3):
    print(f"{Regime_Name_Map[i]}: {Duration[i]:.2f} quarters")

# 3. Preparing data for Chronology and Chart
df_br_gdp['Regime_ID'] = chain_markov_br_gdp_result.smoothed_marginal_probabilities.values.argmax(axis=1)
df_br_gdp['Regime_Name'] = df_br_gdp['Regime_ID'].map(Regime_Name_Map)

# 4. The chart of 3 Regimes
plt.figure(figsize = (15,8))
plt.plot(df_br_gdp.index, df_br_gdp['Index'], color = 'darkblue', lw = 2, label = 'Brazilian GDP')

p = chain_markov_br_gdp_result.smoothed_marginal_probabilities
plt.fill_between(df_br_gdp.index, df_br_gdp['Index'].min(), df_br_gdp['Index'].max(), where = (p[r_rec] > .5), color = 'red', alpha = .3, label = 'Regime: Recession')
plt.fill_between(df_br_gdp.index, df_br_gdp['Index'].min(), df_br_gdp['Index'].max(), where = (p[r_sta] > .5), color = 'orange', alpha = .3, label = 'Regime: Stabilization/Stagnation')
plt.fill_between(df_br_gdp.index, df_br_gdp['Index'].min(), df_br_gdp['Index'].max(), where = (p[r_exp] > .5), color = 'green', alpha = .3, label = 'Regime: Expansion')


plt.xlabel("Quarter")
plt.ylabel("Brazilian GDP - Index")
plt.legend(loc = 'upper left', handlelength = 1.5, frameon = False)
plt.text(0.99, -0.12, 'Source: IBGE - Brazil', transform = plt.gca().transAxes, fontsize = 10, color = 'gray', style = 'italic', horizontalalignment = 'right')
plt.savefig('chart-gdp-three-regimes_en.png', dpi = 300, bbox_inches = 'tight')
plt.show()

# 6. Chronology of the Regimes
df_br_gdp['Regime_ID'] = chain_markov_br_gdp_result.smoothed_marginal_probabilities.idxmax(axis = 1).values

# Mapping and defining the actual regimes
regimes_maps = {r_rec: "Recession", r_sta: "Stabilization/Stagnation", r_exp: "Expansion"}
df_br_gdp['Regime_Name'] = df_br_gdp['Regime_ID'].map(regimes_maps)

# Creating the Chronology Table
regimes_change = df_br_gdp['Regime_Name'] != df_br_gdp['Regime_Name'].shift()
df_br_gdp['Regime_ID'] = regimes_change.cumsum()

chronology = df_br_gdp.reset_index().groupby('Regime_ID').agg(Regime = ('Regime_Name', 'first'), Start = ('Quarter_Date', 'first'),
                                              End = ('Quarter_Date', 'last'), Quarters = ('Regime_Name', 'count'))
chronology['Start'] = pd.PeriodIndex(chronology['Start'], freq='Q').astype(str)
chronology['End'] = pd.PeriodIndex(chronology['End'], freq='Q').astype(str)

print("\nQUARTERLY CHRONOLOGY - THREE REGIMES")
print(chronology.to_string(index = False))

chronology.to_excel('chronology-three-regimes.xlsx', index=False)